# Research Article Classification — Exploration & Model Selection

**Objective:** classify research-paper abstracts into `Science`, `Business`, and `Social Science`.

This notebook uses the cleaned Scopus dataset and compares the six classifiers specified in the assignment:

1. Logistic Regression
2. Decision Tree
3. Multinomial Naive Bayes
4. k-Nearest Neighbors (kNN)
5. Support Vector Machine (Linear SVM)
6. Random Forest

**Methodology:** EDA → NLTK text normalization → TF-IDF unigrams/bigrams → stratified train/test split → stratified cross-validation → model selection using mean Macro-F1 → held-out test evaluation.

> Run the cells from top to bottom. Model training can take several minutes, particularly for kNN and Random Forest. The TF-IDF vectorizer is inside each pipeline, so it is fitted within each CV fold and does not learn vocabulary from that fold's validation data.

## 1. Install and import dependencies

In [ ]:
!pip -q install nltk scikit-learn pandas numpy matplotlib seaborn joblib tqdm

In [ ]:
import os
import re
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import joblib

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
LABEL_ORDER = ["Science", "Business", "Social Science"]
sns.set_theme(style="whitegrid")

## 2. Load the cleaned dataset

Upload `cleaned_scopus_research_abstracts.csv` when prompted. The notebook also accepts a CSV already present in the Colab working directory.

In [ ]:
DATA_PATH = "/content/cleaned_scopus_research_abstracts.csv"

if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Upload the cleaned Scopus CSV to continue.")
    uploaded_name = next(iter(uploaded))
    DATA_PATH = f"/content/{uploaded_name}"

raw_df = pd.read_csv(DATA_PATH)
print("Dataset shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
display(raw_df.head(3))

## 3. Data quality checks and final cleaning

The supplied cleaned dataset should contain `Abstract` and `label`. This section checks missing values, duplicate abstracts, duplicate EIDs and conflicting EID labels. If an EID occurs under multiple labels, all rows for that EID are excluded from this experiment to avoid contradictory training targets. Article and Review records are retained when a document-type column is available.

In [ ]:
def find_column(dataframe, candidates):
    lookup = {str(col).strip().lower(): col for col in dataframe.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

abstract_col = find_column(raw_df, ["Abstract", "abstract", "text"])
label_col = find_column(raw_df, ["label", "category", "class"])
eid_col = find_column(raw_df, ["EID", "eid", "Scopus ID"])
doc_type_col = find_column(raw_df, ["Document Type", "document_type"])

if abstract_col is None or label_col is None:
    raise ValueError("CSV must contain an Abstract column and a label column.")

df = raw_df.copy()

if eid_col is not None:
    conflicting_eids = (
        df.dropna(subset=[eid_col])
          .groupby(eid_col)[label_col]
          .nunique()
    )
    conflicting_eids = conflicting_eids[conflicting_eids > 1].index
    print("Conflicting EIDs:", len(conflicting_eids))
    df = df[~df[eid_col].isin(conflicting_eids)].copy()

if doc_type_col is not None:
    print("Document types before filter:")
    display(df[doc_type_col].value_counts(dropna=False))
    df = df[df[doc_type_col].astype(str).str.strip().str.lower().isin(["article", "review"])].copy()

df[abstract_col] = df[abstract_col].fillna("").astype(str).str.strip()
df[label_col] = df[label_col].fillna("").astype(str).str.strip()
df = df[(df[abstract_col] != "") & (df[label_col].isin(LABEL_ORDER))].copy()

normalized_abstract = df[abstract_col].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
conflicting_abstracts = normalized_abstract.groupby(normalized_abstract).apply(
    lambda values: df.loc[values.index, label_col].nunique()
)
conflicting_abstracts = conflicting_abstracts[conflicting_abstracts > 1].index
df["_normalized_abstract"] = normalized_abstract
df = df[~df["_normalized_abstract"].isin(conflicting_abstracts)].copy()
df = df.drop_duplicates(subset=["_normalized_abstract"]).copy()

df = df.rename(columns={abstract_col: "abstract", label_col: "label"})
df = df[["abstract", "label"]].reset_index(drop=True)

print("Final modeling dataset:", df.shape)
print("\nClass counts:")
display(df["label"].value_counts().rename_axis("Class").to_frame("Samples"))
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing"))


## 4. Exploratory Data Analysis (EDA)

We inspect class balance, abstract length, document length variation and frequent terms. Publication year and source metadata are not used as model features because the stated task is to classify based on abstract text.

In [ ]:
class_counts = df["label"].value_counts().reindex(LABEL_ORDER)

plt.figure(figsize=(8, 5))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Number of Research Abstracts per Class")
plt.xlabel("Class")
plt.ylabel("Number of abstracts")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

df["word_count"] = df["abstract"].str.split().str.len()

plt.figure(figsize=(10, 5))
sns.histplot(
    data=df, x="word_count", hue="label",
    bins=40, element="step", stat="density",
    common_norm=False
)
plt.title("Abstract Length Distribution by Class")
plt.xlabel("Words per abstract")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

display(df.groupby("label")["word_count"].describe().round(1))

In [ ]:
# Inspect the most frequent unigrams before stopword removal.
eda_vectorizer = CountVectorizer(
    stop_words="english",
    max_features=25,
    ngram_range=(1, 1),
    min_df=2
)
term_matrix = eda_vectorizer.fit_transform(df["abstract"])
term_totals = np.asarray(term_matrix.sum(axis=0)).ravel()
top_terms = pd.DataFrame({
    "term": eda_vectorizer.get_feature_names_out(),
    "frequency": term_totals
}).sort_values("frequency", ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_terms.head(20), x="frequency", y="term")
plt.title("Most Frequent Terms Across the Dataset")
plt.xlabel("Frequency")
plt.ylabel("Term")
plt.tight_layout()
plt.show()

display(top_terms.head(20))

## 5. NLTK text cleaning

NLTK is used for regular-expression tokenization, English stopword removal and WordNet lemmatization. This is a lightweight classical NLP pipeline suitable for TF-IDF. It is not a contextual embedding model.

The preprocessing function is stateless with respect to the dataset, so applying it before the split does not estimate vocabulary or IDF statistics. TF-IDF itself is fitted inside the model pipeline on training folds only.

In [ ]:
for package in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*")

def nltk_tokenize(text):
    if text is None:
        return []
    tokens = TOKEN_PATTERN.findall(str(text).lower())
    return [
        LEMMATIZER.lemmatize(token)
        for token in tokens
        if token not in STOP_WORDS and len(token) > 1
    ]

sample_text = df.loc[0, "abstract"]
print("Original:", sample_text[:500])
print("\nCleaned tokens:", nltk_tokenize(sample_text)[:60])

## 6. Train-test split and TF-IDF configuration

We reserve 20% of the data as a final held-out test set. Stratification preserves class proportions. The TF-IDF vectorizer uses unigrams and bigrams, a maximum vocabulary size of 20,000 and sublinear term frequency.

In [ ]:
X = df["abstract"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining class counts:")
display(y_train.value_counts().reindex(LABEL_ORDER).to_frame("Train samples"))
print("\nTesting class counts:")
display(y_test.value_counts().reindex(LABEL_ORDER).to_frame("Test samples"))

def build_vectorizer():
    return TfidfVectorizer(
        tokenizer=nltk_tokenize,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        max_features=20000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        dtype=np.float32
    )

def build_pipeline(classifier):
    return Pipeline([
        ("tfidf", build_vectorizer()),
        ("classifier", classifier)
    ])

## 7. Define the six required classifiers

These are classical ML baselines. Linear SVM, Logistic Regression and Multinomial Naive Bayes are commonly effective for sparse TF-IDF text features. The remaining classifiers are included to satisfy the assignment comparison requirements.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=40,
        min_samples_split=4,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "kNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    ),
    "SVM": LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        min_samples_split=4,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

## 8. Cross-validation and model selection

Each candidate pipeline is evaluated with stratified 3-fold cross-validation on the training partition. Macro-F1 is the primary selection metric because it gives equal weight to each class. The test set is not used to choose the model.

In [ ]:
cv_rows = []
candidate_pipelines = {}

for model_name, classifier in models.items():
    print(f"Cross-validating {model_name}...")
    pipeline = build_pipeline(classifier)

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
        error_score="raise"
    )

    candidate_pipelines[model_name] = pipeline
    cv_rows.append({
        "Model": model_name,
        "CV Macro-F1 Mean": scores.mean(),
        "CV Macro-F1 Std": scores.std()
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    "CV Macro-F1 Mean", ascending=False
).reset_index(drop=True)

display(cv_results.style.format({
    "CV Macro-F1 Mean": "{:.4f}",
    "CV Macro-F1 Std": "{:.4f}"
}))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=cv_results,
    x="CV Macro-F1 Mean",
    y="Model"
)
plt.xlim(0, 1)
plt.title("Model Selection: 3-Fold Cross-Validation Macro-F1")
plt.xlabel("Mean Macro-F1")
plt.ylabel("Classifier")
plt.tight_layout()
plt.show()

best_model_name = cv_results.iloc[0]["Model"]
best_model = candidate_pipelines[best_model_name]
print("Selected model:", best_model_name)

## 9. Fit selected model and evaluate on held-out test set

The selected pipeline is fitted on the full training partition and evaluated once on the untouched test partition. Report accuracy, macro precision, macro recall, macro-F1 and weighted-F1, along with per-class precision/recall/F1.

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
}

print("Selected model:", best_model_name)
display(pd.DataFrame(
    {"Metric": list(test_metrics.keys()), "Score": list(test_metrics.values())}
).style.format({"Score": "{:.4f}"}))

report = classification_report(
    y_test, y_pred,
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).transpose()
display(report_df.round(4))

## 10. Confusion matrix

Rows represent actual classes and columns represent predicted classes. Inspect off-diagonal counts to identify which categories the model confuses.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=LABEL_ORDER)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in LABEL_ORDER],
    columns=[f"Predicted {label}" for label in LABEL_ORDER]
)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.tight_layout()
plt.show()

display(cm_df)

## 11. Save model and evaluation artifacts

The selected fitted pipeline includes the NLTK tokenizer and fitted TF-IDF vectorizer. Keep the `src/text_preprocessing.py` module available when loading the model from the standalone Python project.

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(best_model, ARTIFACT_DIR / "best_model.joblib")
cv_results.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)
report_df.to_csv(ARTIFACT_DIR / "classification_report.csv")
cm_df.to_csv(ARTIFACT_DIR / "confusion_matrix.csv")

results_json = {
    "selected_model": best_model_name,
    "labels": LABEL_ORDER,
    "metrics": {
        "accuracy": float(test_metrics["Accuracy"]),
        "precision_macro": float(test_metrics["Macro Precision"]),
        "recall_macro": float(test_metrics["Macro Recall"]),
        "f1_macro": float(test_metrics["Macro F1"]),
        "f1_weighted": float(test_metrics["Weighted F1"])
    },
    "class_counts": {str(k): int(v) for k, v in df["label"].value_counts().items()},
    "train_class_counts": {str(k): int(v) for k, v in y_train.value_counts().items()},
    "test_class_counts": {str(k): int(v) for k, v in y_test.value_counts().items()},
    "dataset_rows": int(len(df)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "cv_folds": 3,
    "selection_metric": "Mean cross-validation Macro-F1"
}

with open(ARTIFACT_DIR / "results.json", "w", encoding="utf-8") as file:
    json.dump(results_json, file, indent=4)

plt.figure(figsize=(9, 6))
sns.barplot(data=cv_results, x="CV Macro-F1 Mean", y="Model")
plt.xlim(0, 1)
plt.title("Model Comparison: Cross-Validation Macro-F1")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "model_comparison.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=180)
plt.show()

print("Saved artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(path)

## 12. Test a custom abstract

Use this cell to test the selected model on a new abstract. This is a model prediction, not a manually assigned label.

In [ ]:
sample_abstract = (
    "This study investigates the impact of artificial intelligence on "
    "consumer purchasing decisions and business performance."
)

sample_prediction = best_model.predict([sample_abstract])[0]
print("Predicted category:", sample_prediction)

classifier = best_model.named_steps["classifier"]
if hasattr(classifier, "predict_proba"):
    probabilities = best_model.predict_proba([sample_abstract])[0]
    probability_table = pd.DataFrame({
        "Class": best_model.classes_,
        "Model score": probabilities
    }).sort_values("Model score", ascending=False)
    display(probability_table)
else:
    print("This classifier does not provide predict_proba; no probability is reported.")

## 13. Interpretation and reporting checklist

Before writing the report:

- Report the actual class counts after cleaning.
- State the train/test split and cross-validation strategy.
- Use the cross-validation Macro-F1 to explain model selection.
- Report held-out test metrics separately from cross-validation scores.
- Interpret the confusion matrix and per-class metrics.
- Discuss limitations: inherited source labels, interdisciplinary abstracts, class imbalance, Scopus coverage and TF-IDF's lack of contextual semantics.
- Do not describe `predict_proba` scores as calibrated probabilities unless calibration has been evaluated.